In [38]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from datetime import timedelta
from pathlib import Path
from mlforecast import MLForecast
import seaborn as sns

import config
from src.data.reader import read_parquet_pl
import plotly.express as px

pd.set_option("display.max_columns", None)

In [39]:
import importlib
import config
import config.features
importlib.reload(config.features)
importlib.reload(config)

<module 'config' from '/home/njimenez/Workspace/magister/forecast-tfm/notebooks/../config/__init__.py'>

In [40]:
results = pd.read_parquet("../output/experiment_results.parquet")
results.level_id.value_counts()

level_id
10    52
11    52
12    52
1     28
2     28
3     28
4     28
5     28
6     28
7     28
8     28
9     28
Name: count, dtype: int64

In [41]:
results["label"] = (
    results["level_id"]
    .astype(int)
    .astype(str)
    .str.zfill(2)
    + " - "
    + results["level_name"]
)


results["level_id"] = results["level_id"].astype(int)

In [42]:
results.query('level_id == 12')

,level_id,level_name,grain,target,target_type,split,horizon,n_series,n_rows,fit_time_s,wape,bias,wrmsse,model_path,label
356,12,item_store,weekly,sales,sales,valid,4,30490,8476220,15.81,0.350060,-0.051223,0.787889,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store
357,12,item_store,weekly,sales,sales,valid,8,30490,8476220,15.81,0.371808,-0.081033,0.868649,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store
358,12,item_store,weekly,sales,sales,valid,12,30490,8476220,15.81,0.388066,-0.104716,0.903352,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store
359,12,item_store,weekly,sales,sales,valid,16,30490,8476220,15.81,0.402584,-0.127275,0.937474,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store
360,12,item_store,weekly,sales,sales,valid,20,30490,8476220,15.81,0.416302,-0.148545,0.979687,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store
361,12,item_store,weekly,sales,sales,valid,24,30490,8476220,15.81,0.427801,-0.166622,1.005434,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store
362,12,item_store,weekly,sales,sales,valid,28,30490,8476220,15.81,0.439578,-0.182151,1.023950,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store
363,12,item_store,weekly,sales,sales,valid,32,30490,8476220,15.81,0.451246,-0.195875,1.042415,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store
364,12,item_store,weekly,sales,sales,valid,36,30490,8476220,15.81,0.461203,-0.212578,1.056562,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store
365,12,item_store,weekly,sales,sales,valid,40,30490,8476220,15.81,0.471058,-0.229629,1.078928,/home/njimenez/Workspace/magister/forecast-tfm...,12 - item_store


# Análisis de resultados de experimentos


In [43]:
df_plot = results.query('split == "test"').copy()#[
    #(results["level_id"].isin([8,10,12]))
#].copy()

In [44]:
fig = px.scatter(
    df_plot,
    x="label",
    y="wape",
    color="grain",  # daily / weekly
    hover_data=["horizon",'target'],
    width=1200,
    height=300
)

fig.show()

In [45]:
for grain in df_plot["grain"].unique():

    tmp = df_plot[df_plot["grain"] == grain]

    pivot = tmp.pivot_table(
        index="label",
        columns="window",
        values="wape",
        aggfunc="mean"
    )

    fig = px.imshow(
        pivot,
        text_auto=".2f",
        aspect="auto",
        title=grain,
        color_continuous_scale="RdYlGn_r",
        width=600
    )

    fig.show()

KeyError: 'window'

In [ ]:
df_plot = df_plot.sort_values("horizon")

fig = px.line(
    df_plot.sort_values("horizon"),
    x="horizon",
    y="wape",
    color="window",
    facet_row='label',
    facet_col="grain",
    markers=True,
)

fig.update_layout(
    width=1200,
    height=500,
)

fig.show()

In [ ]:
import math

# ── constantes ────────────────────────────────────────────────────────────────
WINDOW_ORDER = ["w1y", "w2y", "w3y", "w4y", "wmax"]
LEVEL_ORDER  = (
    results.sort_values("level_id")["level_name"]
    .drop_duplicates()
    .tolist()
)
GRAINS  = ["daily", "weekly"]
METRICS = ["wape", "wrmsse"]

# ── normalizar horizonte a días (daily × 1, weekly × 7) ──────────────────────
results = results.copy()
results["horizon_days"] = np.where(
    results["grain"] == "daily",
    results["horizon"],
    results["horizon"] * 7,
)

print("horizon_days daily :", sorted(results[results["grain"] == "daily" ]["horizon_days"].unique()))
print("horizon_days weekly:", sorted(results[results["grain"] == "weekly"]["horizon_days"].unique()))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. EFECTO DEL NIVEL JERÁRQUICO
#    ¿El error sube al desagregar? Promediado sobre ventanas y horizontes.
# ══════════════════════════════════════════════════════════════════════════════

metric = "wape"

summary_level = (
    results
    .groupby(["level_id", "level_name", "grain"], as_index=False)
    .agg(**{metric: (metric, "mean")})
    .sort_values("level_id")
)

fig, ax = plt.subplots(figsize=(15, 5))
x     = np.arange(len(LEVEL_ORDER))
width = 0.35

for k, grain in enumerate(GRAINS):
    vals = (
        summary_level[summary_level["grain"] == grain]
        .set_index("level_name")[metric]
        .reindex(LEVEL_ORDER)
    )
    ax.bar(x + (k - 0.5) * width, vals, width, label=grain)

ax.set_xticks(x)
ax.set_xticklabels(LEVEL_ORDER, rotation=40, ha="right")
ax.set_ylabel(metric.upper())
ax.set_title(f"1. {metric.upper()} medio por nivel jerárquico y granularidad")
ax.legend(title="grain")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 2. EFECTO DE LA VENTANA DE ENTRENAMIENTO
#    (a) barras globales por grain   (b) heatmap nivel × ventana
# ══════════════════════════════════════════════════════════════════════════════

summary_win = (
    results
    .groupby(["grain", "window"], as_index=False)
    .agg(**{metric: (metric, "mean")})
)

# 2a. Barras globales
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, grain in zip(axes, GRAINS):
    wins = [w for w in WINDOW_ORDER if w in results[results["grain"] == grain]["window"].values]
    tmp  = (
        summary_win[summary_win["grain"] == grain]
        .set_index("window")[metric]
        .reindex(wins)
        .dropna()
    )
    bars = ax.bar(tmp.index, tmp.values,
                  color=plt.cm.Blues(np.linspace(0.4, 0.85, len(tmp))))
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.set_title(grain)
    ax.set_xlabel("Ventana de entrenamiento")
    ax.set_ylabel(metric.upper())
    ax.set_ylim(0, tmp.values.max() * 1.2)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(f"2a. {metric.upper()} medio por ventana de entrenamiento", fontsize=12)
plt.tight_layout()
plt.show()

# 2b. Heatmap nivel × ventana
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

for ax, grain in zip(axes, GRAINS):
    wins = [w for w in WINDOW_ORDER if w in results[results["grain"] == grain]["window"].unique()]

    pivot = (
        results[results["grain"] == grain]
        .groupby(["level_name", "window"])[[metric]]
        .mean()
        .reset_index()
        .pivot(index="level_name", columns="window", values=metric)
        .reindex(index=LEVEL_ORDER, columns=wins)
    )

    im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(wins)))
    ax.set_xticklabels(wins)
    ax.set_yticks(range(len(LEVEL_ORDER)))
    ax.set_yticklabels(LEVEL_ORDER)
    ax.set_title(grain)

    for i in range(len(LEVEL_ORDER)):
        for j in range(len(wins)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8)

    plt.colorbar(im, ax=ax, shrink=0.8, label=metric.upper())

fig.suptitle(f"2b. Heatmap {metric.upper()}: nivel × ventana de entrenamiento", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. EFECTO DEL HORIZONTE
#    Horizonte normalizado a días para comparar daily vs weekly en el mismo eje.
#    Grid: filas = niveles, columnas = grains.
# ══════════════════════════════════════════════════════════════════════════════

metric = "wape"

plot_df = (
    results
    .groupby(["level_id", "level_name", "grain", "window", "horizon_days"], as_index=False)
    .agg(**{"value": (metric, "mean")})
)

fig, axes = plt.subplots(
    nrows=len(LEVEL_ORDER),
    ncols=2,
    figsize=(14, 3.0 * len(LEVEL_ORDER)),
    sharex=False,
    sharey=False,
)

for i, level in enumerate(LEVEL_ORDER):
    for j, grain in enumerate(GRAINS):
        ax      = axes[i, j]
        df_cell = plot_df[(plot_df["level_name"] == level) & (plot_df["grain"] == grain)]
        wins    = [w for w in WINDOW_ORDER if w in df_cell["window"].values]

        for w in wins:
            tmp = df_cell[df_cell["window"] == w].sort_values("horizon_days")
            ax.plot(tmp["horizon_days"], tmp["value"], marker="o", markersize=4, label=w)

        ax.set_title(f"{level} – {grain}", fontsize=9)
        ax.set_xlabel("Horizonte (días)", fontsize=8)
        ax.set_ylabel(metric.upper(), fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.3)

handles, labels = next(
    (ax.get_legend_handles_labels() for ax in axes.flatten()
     if ax.get_legend_handles_labels()[0]),
    ([], []),
)
fig.legend(handles, labels, title="window", loc="upper center",
           ncol=len(WINDOW_ORDER), bbox_to_anchor=(0.5, 1.005))
fig.suptitle(f"3. {metric.upper()} por horizonte (días), nivel y ventana de entrenamiento", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. VISTA COMPACTA: GRAIN × WINDOW POR NIVEL
#    Un panel por nivel; daily (—) vs weekly (--), todas las ventanas.
#    Permite ver rápidamente cuál combinación grain+window es mejor por nivel.
# ══════════════════════════════════════════════════════════════════════════════

metric = "wape"

plot_df = (
    results
    .groupby(["level_id", "level_name", "grain", "window", "horizon_days"], as_index=False)
    .agg(**{"value": (metric, "mean")})
)

ncols    = 3
nrows    = math.ceil(len(LEVEL_ORDER) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.5 * nrows), sharex=False, sharey=False)
axes_flat = axes.flatten()

for ax, level in zip(axes_flat, LEVEL_ORDER):
    df_l   = plot_df[plot_df["level_name"] == level]
    combos = (
        df_l[["grain", "window"]]
        .drop_duplicates()
        .sort_values(["grain", "window"])
        .itertuples(index=False, name=None)
    )
    for grain, window in combos:
        tmp = (
            df_l[(df_l["grain"] == grain) & (df_l["window"] == window)]
            .sort_values("horizon_days")
        )
        ax.plot(
            tmp["horizon_days"], tmp["value"],
            marker="o", markersize=4,
            linestyle="-" if grain == "daily" else "--",
            label=f"{grain}|{window}",
        )
    ax.set_title(level, fontsize=10)
    ax.set_xlabel("Horizonte (días)", fontsize=8)
    ax.set_ylabel(metric.upper(), fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3)

for ax in axes_flat[len(LEVEL_ORDER):]:
    ax.axis("off")

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, title="grain | window", loc="upper center",
           ncol=5, bbox_to_anchor=(0.5, 1.01), fontsize=8)
fig.suptitle(f"4. {metric.upper()} por horizonte, grain y window — un panel por nivel", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. ANÁLISIS DEL SESGO (BIAS)
#    Negativo → subestimación sistemática.
#    (a) por nivel y grain  (b) por horizonte y ventana
# ══════════════════════════════════════════════════════════════════════════════

# 5a. Bias por nivel y grain
summary_bias = (
    results
    .groupby(["level_id", "level_name", "grain"], as_index=False)
    .agg(bias=("bias", "mean"))
    .sort_values("level_id")
)

fig, ax = plt.subplots(figsize=(15, 5))
x     = np.arange(len(LEVEL_ORDER))
width = 0.35

for k, grain in enumerate(GRAINS):
    vals = (
        summary_bias[summary_bias["grain"] == grain]
        .set_index("level_name")["bias"]
        .reindex(LEVEL_ORDER)
    )
    ax.bar(x + (k - 0.5) * width, vals, width, label=grain)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(LEVEL_ORDER, rotation=40, ha="right")
ax.set_ylabel("Bias medio")
ax.set_title("5a. Bias medio por nivel y granularidad  (negativo = subestimación)")
ax.legend(title="grain")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# 5b. Bias por horizonte y ventana
bias_hor = (
    results
    .groupby(["grain", "window", "horizon_days"], as_index=False)
    .agg(bias=("bias", "mean"))
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, grain in zip(axes, GRAINS):
    df_g = bias_hor[bias_hor["grain"] == grain]
    wins = [w for w in WINDOW_ORDER if w in df_g["window"].values]
    for w in wins:
        tmp = df_g[df_g["window"] == w].sort_values("horizon_days")
        ax.plot(tmp["horizon_days"], tmp["bias"], marker="o", markersize=4, label=w)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"Bias vs horizonte – {grain}")
    ax.set_xlabel("Horizonte (días)")
    ax.set_ylabel("Bias medio")
    ax.legend(title="window")
    ax.grid(alpha=0.3)

fig.suptitle("5b. Bias por horizonte y ventana de entrenamiento", fontsize=12)
plt.tight_layout()
plt.show()
